![image](car.jpeg)

**Car-ing is sharing**, an auto dealership company for car sales and rental, is taking their services to the next level thanks to **Large Language Models (LLMs)** to help prototype a chatbot app that addresses diverse inquiries using LLMs.

The solution should receive textual prompts and use a variety of pre-trained Hugging Face LLMs to respond to a series of tasks, e.g. classifying the sentiment in a car’s text review, answering a customer question, summarizing or translating text, etc.


## Task 1:

Use a pre-trained LLM to classify the sentiment of the five car reviews in the car_reviews.csv dataset, and evaluate the classification accuracy and F1 score of predictions.
- Store the model outputs in predicted_labels, then extract the labels and map them onto a list of {0,1} integer binary labels called predictions.
- Store the calculated metrics in accuracy_result and f1_result.

In [212]:
# Import necessary packages
import pandas as pd
import torch

from transformers import logging
logging.set_verbosity(logging.WARNING)

In [213]:
# Start your code here!
from transformers import pipeline
model = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

Device set to use cpu


In [214]:
df = pd.read_csv("data/car_reviews.csv", delimiter=";")
df.head()

,Review,Class
0,I am very satisfied with my 2014 Nissan NV SL....,POSITIVE
1,The car is fine. It's a bit loud and not very ...,NEGATIVE
2,"My first foreign car. Love it, I would buy ano...",POSITIVE
3,I've come across numerous reviews praising the...,NEGATIVE
4,I've been dreaming of owning an SUV for quite ...,POSITIVE


In [215]:
reviews = df.Review.tolist()
sentiment = df.Class.tolist()

In [216]:
reviews

['I am very satisfied with my 2014 Nissan NV SL. I use this van for my business deliveries and personal use. Camping, road trips, etc. We dont have any children so I store most of the seats in my warehouse. I wanted the passenger van for the rear air conditioning. We drove our van from Florida to California for a Cross Country trip in 2014. We averaged about 18 mpg. We drove thru a lot of rain and It was a very comfortable and stable vehicle. The V8 Nissan Titan engine is a 500k mile engine. It has been tested many times by delivery and trucking companies. This is why Nissan gives you a 5 year or 100k mile bumper to bumper warranty. Many people are scared about driving this van because of its size. But with front and rear sonar sensors, large mirrors and the back up camera. It is easy to drive. The front and rear sensors also monitor the front and rear sides of the bumpers making it easier to park close to objects. Our Nissan NV is a Tow Monster. It pulls our 5000 pound travel trailer 

In [217]:
predicted_labels = model(reviews)
predicted_labels

[{'label': 'POSITIVE', 'score': 0.929397702217102},
 {'label': 'POSITIVE', 'score': 0.8654273152351379},
 {'label': 'POSITIVE', 'score': 0.9994640946388245},
 {'label': 'NEGATIVE', 'score': 0.9935314059257507},
 {'label': 'POSITIVE', 'score': 0.9986565113067627}]

In [218]:
for review, prediction, label in zip(reviews, predicted_labels, sentiment):
    print(f"{review}: {prediction['label']}")

I am very satisfied with my 2014 Nissan NV SL. I use this van for my business deliveries and personal use. Camping, road trips, etc. We dont have any children so I store most of the seats in my warehouse. I wanted the passenger van for the rear air conditioning. We drove our van from Florida to California for a Cross Country trip in 2014. We averaged about 18 mpg. We drove thru a lot of rain and It was a very comfortable and stable vehicle. The V8 Nissan Titan engine is a 500k mile engine. It has been tested many times by delivery and trucking companies. This is why Nissan gives you a 5 year or 100k mile bumper to bumper warranty. Many people are scared about driving this van because of its size. But with front and rear sonar sensors, large mirrors and the back up camera. It is easy to drive. The front and rear sensors also monitor the front and rear sides of the bumpers making it easier to park close to objects. Our Nissan NV is a Tow Monster. It pulls our 5000 pound travel trailer li

In [219]:
import evaluate
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

In [220]:
for i in predicted_labels:
    print(i['label'])

POSITIVE
POSITIVE
POSITIVE
NEGATIVE
POSITIVE


In [221]:
references = [1 if label == "POSITIVE" else 0 for label in sentiment]
predictions = [1 if label['label'] == "POSITIVE" else 0 for label in predicted_labels]

In [222]:
accuracy_result = accuracy.compute(references=references, predictions=predictions)
f1_result = f1.compute(references=references, predictions=predictions)

In [223]:
accuracy_result = accuracy_result['accuracy']
f1_result = f1_result["f1"]

## Task 2:

The company is recently attracting customers from Spain. Extract and pass the first two sentences of the first review in the dataset to an English-to-Spanish translation LLM. Calculate the BLEU score to assess translation quality, using the content in reference_translations.txt as references.
- Store the translated text generated by the LLM in translated_review.
- Store the BLEU score metric result in bleu_score.

In [224]:
first = reviews[0]
first

'I am very satisfied with my 2014 Nissan NV SL. I use this van for my business deliveries and personal use. Camping, road trips, etc. We dont have any children so I store most of the seats in my warehouse. I wanted the passenger van for the rear air conditioning. We drove our van from Florida to California for a Cross Country trip in 2014. We averaged about 18 mpg. We drove thru a lot of rain and It was a very comfortable and stable vehicle. The V8 Nissan Titan engine is a 500k mile engine. It has been tested many times by delivery and trucking companies. This is why Nissan gives you a 5 year or 100k mile bumper to bumper warranty. Many people are scared about driving this van because of its size. But with front and rear sonar sensors, large mirrors and the back up camera. It is easy to drive. The front and rear sensors also monitor the front and rear sides of the bumpers making it easier to park close to objects. Our Nissan NV is a Tow Monster. It pulls our 5000 pound travel trailer l

In [225]:
translate_model = pipeline("translation", model="Helsinki-NLP/opus-mt-en-es")
translated_review = translate_model(first)

Device set to use cpu


In [226]:
translated_review = translated_review[0]["translation_text"]

In [227]:
translated_review = translated_review[:115]

In [228]:
file = open("data/reference_translations.txt")
new = file.readlines()
references = [i.strip() for i in new][0]
references

'Estoy muy satisfecho con mi Nissan NV SL 2014. Utilizo esta camioneta para mis entregas comerciales y uso personal.'

In [229]:
len(references)

115

In [230]:
bleu = evaluate.load("bleu")
bleu_score = bleu.compute(predictions=[translated_review], references=[references])
#bleu_score = bleu_score["bleu"]

## Task 3:

The 2nd review in the dataset emphasizes brand aspects. Load an extractive QA LLM such as "deepset/minilm-uncased-squad2" to formulate the question "What did he like about the brand?" and obtain an answer.
- Use question and context for the two variables containing the LLM inputs: question and context.
- Store the actual text answer in answer

In [231]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

In [232]:
model_QA = AutoModelForQuestionAnswering.from_pretrained("deepset/minilm-uncased-squad2")
tokenizer = AutoTokenizer.from_pretrained("deepset/minilm-uncased-squad2")

Some weights of the model checkpoint at deepset/minilm-uncased-squad2 were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [233]:
context = reviews[1]
context

"The car is fine. It's a bit loud and not very powerful. On one hand, compared to its peers, the interior is well-built. The transmission failed a few years ago, and the dealer replaced it under warranty with no issues. Now, about 60k miles later, the transmission is failing again. It sounds like a truck, and the issues are well-documented. The dealer tells me it is normal, refusing to do anything to resolve the issue. After owning the car for 4 years, there are many other vehicles I would purchase over this one. Initially, I really liked what the brand is about: ride quality, reliability, etc. But I will not purchase another one. Despite these concerns, I must say, the level of comfort in the car has always been satisfactory, but not worth the rest of issues found."

In [234]:
question = "What did he like about the brand?"
token = tokenizer(question, context, return_tensors="pt")

In [235]:
with torch.no_grad():
    outputs = model_QA(**token)

In [236]:
start_idx = torch.argmax(outputs.start_logits)
end_idx = torch.argmax(outputs.end_logits) + 1
answer = token["input_ids"][0][start_idx:end_idx]
answer

tensor([ 4536,  3737,  1010, 15258])

In [237]:
answer = tokenizer.decode(answer)
answer

'ride quality, reliability'

## Task 4:

Summarize the last review in the dataset, into approximately 50-55 tokens long. Store it in the variable summarized_text.

In [238]:
last = reviews[-1]
last

"I've been dreaming of owning an SUV for quite a while, but I've been driving cars that were already paid for during an extended period. I ultimately made the decision to transition to a brand-new car, which, of course, involved taking on new payments. However, given that I don't drive extensively, I was inclined to avoid a substantial financial commitment. The Nissan Rogue provides me with the desired SUV experience without burdening me with an exorbitant payment; the financial arrangement is quite reasonable. Handling and styling are great; I have hauled 12 bags of mulch in the back with the seats down and could have held more. I am VERY satisfied overall. I find myself needing to exercise extra caution when making lane changes, particularly owing to the blind spots resulting from the small side windows situated towards the rear of the vehicle. To address this concern, I am actively engaged in making adjustments to my mirrors and consciously reducing the frequency of lane changes. Th

In [239]:
model_sum = pipeline("summarization", model="cnicu/t5-small-booksum")
result = model_sum(last, max_length=55)
result

Device set to use cpu


[{'summary_text': 'the Nissan Rogue provides me with the desired SUV experience without burdening me with an exorbitant payment; the financial arrangement is quite reasonable. I have hauled 12 bags of mulch in the back with the seats down and could have held more. I find'}]

In [240]:
summarized_text = result[0]["summary_text"]
summarized_text

'the Nissan Rogue provides me with the desired SUV experience without burdening me with an exorbitant payment; the financial arrangement is quite reasonable. I have hauled 12 bags of mulch in the back with the seats down and could have held more. I find'